# Image dataset loading and augmentation utilities for CIFAR-10.



In [ ]:
import sys
from pathlib import Path
import plotly.express as px

ROOT = Path(__file__).resolve().parents[2] if "__file__" in dir() else Path("/Users/manavdahra/workspace/sam-opt-research")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from matplotlib.pylab import shuffle
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

class _HFCifar10Dataset(Dataset):
    """Thin wrapper around a HuggingFace CIFAR-10 split.

    Each HF sample is {'img': PIL.Image, 'label': int}.
    """

    def __init__(self, hf_split, transform):
        self._data = hf_split
        self._transform = transform

    def __len__(self) -> int:
        return len(self._data)

    def __getitem__(self, idx: int):
        sample = self._data[idx]
        img = sample["img"].convert("RGB")
        label = sample["label"]
        return self._transform(img), label


def get_cifar10_dataset(
    data_dir: str = "./data",
    resize: int | None = None,
) -> tuple[Dataset, Dataset]:
    """Return (train_dataset, test_dataset) for CIFAR-10.

    Loads from HuggingFace (uoft-cs/cifar10) and caches under data_dir.

    Args:
        data_dir: Directory for HuggingFace dataset cache.
        resize: If given, resize images to this square size (e.g. 224 for ViT).
    """
    base_transforms = []
    if resize is not None:
        base_transforms.append(transforms.Resize((resize, resize)))

    train_transform = transforms.Compose(base_transforms)

    test_transform = transforms.Compose(base_transforms)

    hf_ds = load_dataset(
        "uoft-cs/cifar10",
        cache_dir=data_dir,
    )

    train_dataset = _HFCifar10Dataset(hf_ds["train"], train_transform)
    test_dataset = _HFCifar10Dataset(hf_ds["test"], test_transform)

    return train_dataset, test_dataset


In [ ]:
train_1, test_1 = get_cifar10_dataset("./data")
train_2, test_2 = get_cifar10_dataset("./data", resize=224)

In [ ]:
import matplotlib.pyplot as plt

CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]

# Collect one example per class from the raw dataset (PIL images, no transform)
seen = {}
for sample in train_2._data:
    label = sample["label"]
    if label not in seen:
        seen[label] = sample["img"].convert("RGB")
    if len(seen) == 10:
        break

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for ax, (label, img) in zip(axes, sorted(seen.items())):
    ax.imshow(img)
    ax.set_title(CIFAR10_CLASSES[label], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()
